# 04 · Baseline gợi ý sản phẩm (Tuần 6)
Notebook này chỉ **khám phá và chạy baseline**; toàn bộ logic nằm trong `models/src/`.

- Bài toán, nhãn, cách chia dữ liệu và metric: xem `reports/problem-statement.md`.
- Baseline: (1) top bán chạy toàn chuỗi 30 ngày; (2) top bán chạy theo giới tính của bé (suy ra từ lịch sử mua) và theo mùa.
- Chạy từ thư mục gốc repo, hoặc đặt `MODELS_DIR` bên dưới cho đúng đường dẫn tới thư mục `models/`.

In [ ]:
import sys, pathlib
MODELS_DIR = pathlib.Path("../models").resolve()          # đổi nếu notebook nằm ở nơi khác
sys.path.insert(0, str(MODELS_DIR))
import numpy as np, pandas as pd
from IPython.display import display
from src import config as C, features as F, models as M
from src.split import build_ctx
from src.metrics import evaluate, summarize
pd.set_option("display.width", 200); pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

## 1. Dữ liệu và mã không phải hàng hóa
Đọc từ PostgreSQL (cache vào `models/data/`). Đặt `use_cache=False` để đọc lại từ database.

In [ ]:
raw = F.load_raw(use_cache=True)
blacklist, stats = F.find_non_merch(raw["item_stats"], int(raw["meta"].n_inv_total[0]), C.NON_MERCH_MANUAL)
print(f"Loại {len(blacklist)} mã không phải hàng hóa (túi, quà tặng, voucher...)")
display(stats[stats.item.isin(blacklist)].sort_values("qty", ascending=False).head(10)[["item", "qty", "rev", "gia_tb", "ty_le_hd_pct"]])
inter = raw["inter"][~raw["inter"].item.isin(blacklist)]
print(f"{len(inter):,} dòng (khách, mẫu, hóa đơn) | {inter.customer_id.nunique():,} khách | {inter.item.nunique():,} mẫu")

## 2. Tập huấn luyện và tập kiểm tra
Ngày chốt T chung cho mọi khách: huấn luyện chỉ dùng dữ liệu < T; kiểm tra gồm khách có lịch sử trước T và có mua trong [T, hết kỳ); nhãn là các sản phẩm của **hóa đơn cuối cùng** của khách trong kỳ đó.

In [ ]:
age, vocab = F.item_age_table(raw["sku"]), F.build_vocab(raw["items"])
ctx_val = build_ctx(inter, raw["items"], age, vocab, *C.VAL_FOLDS[-1])
ctx_test = build_ctx(inter, raw["items"], age, vocab, *C.TEST)
print("validation:", ctx_val.summary()); print("test      :", ctx_test.summary())

## 3. Baseline

In [ ]:
rows = []
for ctx, tag in [(ctx_val, "validation"), (ctx_test, "test")]:
    for m in [M.PopGlobal(30), M.PopSegment(30)]:
        m.fit(ctx)
        for sf in (False, True):
            df = summarize(evaluate(ctx, m, size_filter=sf), m.name + (" + lọc size" if sf else ""))
            df.insert(0, "tap", tag); rows.append(df)
base = pd.concat(rows, ignore_index=True)
base.to_csv(C.REPORTS / "baseline_ket_qua.csv", index=False)
base[base.nhom == "tat_ca"][["tap", "mo_hinh", "so_khach", "precision@10", "recall@10", "ndcg@10", "hit_rate@10", "coverage"]]

## 4. Nhận xét (điền sau khi chạy)
- Baseline theo giới tính/mùa hơn baseline toàn chuỗi bao nhiêu?
- Lọc size làm đổi metric thế nào (đổi lấy việc không gợi ý sai size)?
- Trần Recall của bài toán là bao nhiêu (xem dòng `trần Recall` ở trên)? Vì sao Recall@10 không thể gần 1?